# Word Embeddings — Teoría y Práctica

Este cuaderno recorre las ideas detrás de los word embeddings y las conecta
con los scripts de este proyecto: aritmética vectorial, agrupamiento de tweets y búsqueda semántica.

In [ ]:
import sys
from pathlib import Path

ROOT = Path(".").resolve()   # class_pip/
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

---
## 1. ¿Qué Son los Word Embeddings?

El NLP tradicional representaba las palabras como **vectores one-hot** — un vector con un único `1`
en la posición correspondiente a la palabra, y ceros en todo lo demás.

```
king  → [1, 0, 0, 0, ...]
queen → [0, 1, 0, 0, ...]
man   → [0, 0, 1, 0, ...]
```

Problemas con one-hot:
- Cada palabra está a la misma distancia de todas las demás (sin noción de similitud)
- El vocabulario es enorme → vectores muy dispersos y de alta dimensionalidad

**Los word embeddings** mapean cada palabra a un vector denso de baja dimensionalidad (por ejemplo, 100 o 300 dimensiones)
de modo que **las palabras similares quedan cerca entre sí** en ese espacio.

> *"You shall know a word by the company it keeps."* — J.R. Firth (1957)

Esta es la **hipótesis distribucional**: las palabras que aparecen en contextos similares tienden a tener significados similares.
Los modelos de embeddings aprenden a partir de patrones de co-ocurrencia en grandes corpus.

---
## 2. Cómo Funciona GloVe

**GloVe** (Global Vectors for Word Representation, Pennington et al. 2014) construye una
**matriz de co-ocurrencia** sobre un corpus amplio: la entrada `X[i,j]` cuenta cuántas veces la palabra `j`
aparece en el contexto de la palabra `i`.

Luego aprende vectores `u_i`, `v_j` tales que:

$$u_i \cdot v_j \approx \log X_{ij}$$

Esto fuerza a que el **producto punto** de dos vectores de palabras refleje la frecuencia con que co-ocurren —
por lo que las palabras que comparten contextos terminan siendo geométricamente cercanas.

En este proyecto usamos dos variantes de GloVe:
| Modelo | Entrenado en | Dim | Mejor para |
|---|---|---|---|
| `glove-wiki-gigaword-100` | Wikipedia + Gigaword | 100 | Texto general |
| `glove-twitter-100` | Twitter (2B tweets) | 100 | Tweets, lenguaje informal |

In [ ]:
import gensim.downloader as api

# Usamos la variante de Twitter porque nuestro dataset es de tweets
wv = api.load("glove-twitter-100")
print(f"Tamaño del vocabulario: {len(wv)}")
print(f"Dimensión del vector  : {wv.vector_size}")

---
## 3. Aritmética Vectorial

Una de las propiedades más llamativas de los embeddings es que las **direcciones en el espacio vectorial
tienen significado**. El ejemplo clásico:

$$\vec{king} - \vec{man} + \vec{woman} \approx \vec{queen}$$

El vector `man - woman` codifica la **dirección de género**. Proyectar cualquier palabra sobre
esa dirección nos indica qué tan "masculina" es en relación a qué tan "femenina" es.

Esta es la misma idea que se usa en `embeddings.py` en la raíz de este proyecto.

In [ ]:
import numpy as np

# Analogía clásica: king - man + woman ≈ queen
result = wv.most_similar(positive=["king", "woman"], negative=["man"], topn=3)
print("king - man + woman:")
for word, score in result:
    print(f"  {word:<12} {score:.3f}")

In [ ]:
# Dirección de género: ¿qué tan "masculina" vs "femenina" es cada palabra?
gender_axis = wv["man"] - wv["woman"]

words = ["king", "queen", "prince", "princess", "he", "she", "boy", "girl"]
print(f"{'palabra':<12} {'proyección':>10}  (positivo = más masculino)")
print("-" * 38)
for w in words:
    proj = np.dot(wv[w], gender_axis)
    print(f"{w:<12} {proj:>10.3f}")

---
## 4. Visualización del Espacio de Embeddings con PCA

Los vectores de embeddings viven en 100 dimensiones — imposible de visualizar directamente.
**PCA** (Análisis de Componentes Principales) los proyecta sobre las 2 direcciones que
preservan la mayor varianza, dándonos un mapa en 2D.

> ⚠️ El mapa en 2D es una proyección distorsionada. Las distancias en 2D no reflejan perfectamente
> las distancias en 100D — pero la estructura general (clusters, vecindarios) suele ser visible.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

words = ["king", "queen", "man", "woman", "dog", "cat", "car", "plane",
         "happy", "sad", "love", "hate"]

X = np.array([wv[w] for w in words])
X_2d = PCA(n_components=2).fit_transform(X)

_, ax = plt.subplots(figsize=(8, 5))
ax.scatter(X_2d[:, 0], X_2d[:, 1], s=40)
for i, w in enumerate(words):
    ax.annotate(w, (X_2d[i, 0] + 0.01, X_2d[i, 1] + 0.01), fontsize=10)
ax.set_title("Vectores de palabras proyectados a 2D (PCA)")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
plt.tight_layout()
plt.show()

Nótese que las palabras semánticamente relacionadas (man/woman/king/queen, dog/cat, car/plane)
se agrupan juntas — incluso en esta proyección 2D con pérdida de información.

---
## 5. De Palabras a Oraciones: Mean Pooling

GloVe nos da un vector por **palabra**. Para representar un **tweet** completo necesitamos
agregar esos vectores en uno solo.

El enfoque más sencillo es el **mean pooling**: promediar los vectores de todos los tokens del texto.

$$\vec{tweet} = \frac{1}{|T|} \sum_{w \in T} \vec{w}$$

Esto es lo que hacen tanto `tweet_clusters.py` como `tweet_search.py`.
Es rápido y sorprendentemente efectivo, aunque descarta por completo el orden de las palabras.

In [ ]:
from preprocessing import cleaner

def mean_vector(text):
    tokens = cleaner(text)
    vecs   = [wv[w] for w in tokens if w in wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(wv.vector_size)

tweet = "I'm feeling really sad today"
vec   = mean_vector(tweet)
print(f"Tweet  : {tweet!r}")
print(f"Vector : shape={vec.shape}, primeras 5 dims={vec[:5].round(3)}")

---
## 6. Agrupamiento de Tweets

Una vez que cada tweet es un vector, podemos aplicar algoritmos estándar de ML.
**KMeans** agrupa los tweets en `k` clusters minimizando la distancia intra-cluster.

Los tweets cuyos vectores GloVe promedio son cercanos → vocabulario y temas similares → mismo cluster.

> ⚠️ Las etiquetas de palabras principales pueden verse ruidosas (las stop words dominan) si el limpiador no
> las elimina. Usa `SpacyCleaner(remove_stops=True)` de `preprocessing.py` para obtener etiquetas más limpias.

In [ ]:
from collections import Counter
from sklearn.cluster import KMeans
from dataset import TwitterData

N_TWEETS   = 500
N_CLUSTERS = 4
TOP_WORDS  = 5

tweets           = TwitterData.test_only().test_texts[:N_TWEETS]
tokens_per_tweet = [cleaner(t) for t in tweets]
X                = np.array([mean_vector(t) for t in tweets])

labels = KMeans(n_clusters=N_CLUSTERS, random_state=42, n_init="auto").fit_predict(X)

print("Palabras clave por cluster:")
for c in range(N_CLUSTERS):
    all_tokens = [t for i, toks in enumerate(tokens_per_tweet)
                  if labels[i] == c for t in toks]
    kw = ", ".join(w for w, _ in Counter(all_tokens).most_common(TOP_WORDS))
    print(f"  cluster {c}: {kw}")

In [ ]:
X_2d   = PCA(n_components=2).fit_transform(X)
colors = plt.cm.tab10.colors

_, ax = plt.subplots(figsize=(9, 5))
for c in range(N_CLUSTERS):
    mask = labels == c
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1],
               color=colors[c], label=f"cluster {c}", alpha=0.6, s=20)
ax.set_title("Clusters de tweets — GloVe + KMeans (PCA 2D)")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.legend()
plt.tight_layout()
plt.show()

---
## 7. Búsqueda por Similitud Semántica

Dado un tweet de consulta, queremos encontrar los tweets más **semánticamente similares** en nuestro corpus
sin ningún entrenamiento — simplemente comparando sus vectores.

Usamos la **similitud coseno**, que mide el ángulo entre dos vectores:

$$\text{sim}(A, B) = \frac{A \cdot B}{\|A\| \|B\|}$$

- `1.0` → dirección idéntica (muy similares)
- `0.0` → perpendiculares (sin relación)
- `-1.0` → direcciones opuestas

Preferimos el coseno sobre la distancia euclidiana porque es **invariante a la magnitud**:
un tweet corto y uno largo sobre el mismo tema pueden seguir obteniendo una puntuación de 1.0.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

QUERY = "I'm not happy"
TOP_N = 10

q_vec   = mean_vector(QUERY).reshape(1, -1)
scores  = cosine_similarity(q_vec, X)[0]   # devuelve (1, N) → tomamos la fila 0
ranking = np.argsort(-scores)              # negamos → argsort ascendente = descendente por score

print(f"Consulta: {QUERY!r}\n")
for rank, idx in enumerate(ranking[:TOP_N], 1):
    print(f"  {rank:>2}. [{scores[idx]:.3f}] {tweets[idx]}")

In [ ]:
coords      = PCA(n_components=2).fit_transform(np.vstack([X, q_vec]))
X_2d, q_2d = coords[:-1], coords[-1]

top_mask = np.zeros(len(X), dtype=bool)
top_mask[ranking[:TOP_N]] = True

_, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_2d[~top_mask, 0], X_2d[~top_mask, 1], c="lightgrey", s=15, label="tweets")
sc = ax.scatter(X_2d[top_mask, 0], X_2d[top_mask, 1],
                c=scores[top_mask], cmap="YlOrRd", s=60, label=f"top {TOP_N}")
ax.scatter(*q_2d, marker="*", s=300, color="steelblue", label="consulta", zorder=3)
plt.colorbar(sc, ax=ax, label="similitud coseno")
ax.set_title(f"Búsqueda semántica — consulta: {QUERY!r}")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.legend(); plt.tight_layout(); plt.show()

print("\n⚠️  La barra de color es la señal fiable — la distancia espacial en 2D está distorsionada por PCA.")

---
## 8. El Problema de los Antónimos

Puede notarse que al buscar `"I'm not happy"` se devuelven algunos tweets **positivos**.
Esto no es un error — es una propiedad fundamental de GloVe.

GloVe aprende a partir de la **co-ocurrencia**: las palabras que aparecen en los mismos contextos obtienen vectores similares.
`happy` y `sad` ambas aparecen cerca de `"feeling"`, `"so"`, `"today"`, `"very"` — por lo que terminan siendo
**geométricamente cercanas**, a pesar de ser opuestos semánticos.

```
GloVe distance:  happy ≈ sad  ✗  (mismo contexto)
Human judgement: happy ≠ sad  ✓  (significados opuestos)
```

Esto se denomina la **confusión similitud/relación**: GloVe captura la *relación temática*
(ambas palabras tratan sobre emociones) pero no la *polaridad semántica* (positivo vs negativo).

In [ ]:
# Demostración: happy y sad están más cerca de lo esperado
sim_happy_sad  = wv.similarity("happy", "sad")
sim_happy_good = wv.similarity("happy", "good")
sim_happy_car  = wv.similarity("happy", "car")

print(f"happy ↔ sad  : {sim_happy_sad:.3f}")
print(f"happy ↔ good : {sim_happy_good:.3f}")
print(f"happy ↔ car  : {sim_happy_car:.3f}")
print("\n'happy' y 'sad' están mucho más cerca que 'happy' y 'car'.")

---
## 9. Búsqueda Ajustada por Léxico con VADER

Un **léxico de sentimientos** es un diccionario elaborado manualmente que asigna a palabras (y frases)
puntuaciones de sentimiento — construido por personas, no aprendido a partir de datos.

**VADER** (Valence Aware Dictionary and sEntiment Reasoner) es un léxico popular diseñado
específicamente para redes sociales. Contiene ~7,500 entradas y maneja:
- Negación: `"not happy"` → negativo
- Intensificadores: `"very good"` > `"good"`
- Puntuación/mayúsculas: `"GREAT!!!"` > `"great"`

Devuelve una **puntuación compuesta** en `[-1, +1]`.

### Combinando GloVe + VADER

Ponderamos la similitud coseno según qué tan bien el sentimiento del tweet **coincide** con el de la consulta:

$$\text{combined} = \text{cosine\_sim} \times \underbrace{\left(1 - \frac{|s_q - s_t|}{2}\right)}_{\text{alineación de sentimiento} \in [0,1]}$$

Cuando la consulta y el tweet tienen la misma polaridad → alineación ≈ 1 (sin penalización).  
Cuando son opuestos → alineación ≈ 0 (la puntuación colapsa).

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

vader = SentimentIntensityAnalyzer()

def sentiment_score(text):
    return vader.polarity_scores(text)["compound"]

# VADER maneja la negación correctamente
examples = ["I'm not happy", "I'm happy", "This is terrible", "This is great!"]
for ex in examples:
    print(f"  {sentiment_score(ex):+.3f}  {ex!r}")

In [ ]:
q_sentiment      = sentiment_score(QUERY)
tweet_sentiments = np.array([sentiment_score(t) for t in tweets])
sentiment_sim    = 1 - np.abs(q_sentiment - tweet_sentiments) / 2

combined_scores  = scores * sentiment_sim
combined_ranking = np.argsort(-combined_scores)

print(f"Consulta: {QUERY!r}  (VADER: {q_sentiment:+.3f})\n")
for rank, idx in enumerate(combined_ranking[:TOP_N], 1):
    s_tweet = tweet_sentiments[idx]
    print(f"  {rank:>2}. [cos={scores[idx]:.3f} | vader={s_tweet:+.3f} | combined={combined_scores[idx]:.3f}] {tweets[idx]}")

In [ ]:
top_mask_adj = np.zeros(len(X), dtype=bool)
top_mask_adj[combined_ranking[:TOP_N]] = True

_, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_2d[~top_mask_adj, 0], X_2d[~top_mask_adj, 1], c="lightgrey", s=15, label="tweets")
sc = ax.scatter(X_2d[top_mask_adj, 0], X_2d[top_mask_adj, 1],
                c=combined_scores[top_mask_adj], cmap="YlOrRd", s=60, label=f"top {TOP_N}")
ax.scatter(*q_2d, marker="*", s=300, color="steelblue", label="consulta", zorder=3)
plt.colorbar(sc, ax=ax, label="puntuación combinada")
ax.set_title(f"Búsqueda ajustada por léxico — consulta: {QUERY!r}")
ax.set_xlabel("PC 1"); ax.set_ylabel("PC 2")
ax.legend(); plt.tight_layout(); plt.show()

---
## Resumen

| Concepto | Idea clave | Script |
|---|---|---|
| Word embeddings | Vectores densos aprendidos a partir de co-ocurrencia | `embeddings.py` |
| Aritmética vectorial | Las direcciones codifican significado (género, sentimiento) | `embeddings.py` |
| Mean pooling | Promedio de vectores de palabras → vector de oración | `tweet_search.py` |
| Similitud coseno | Ángulo entre vectores, invariante a la magnitud | `tweet_search.py` |
| Problema de antónimos | GloVe confunde relación temática con polaridad | `tweet_search.py` |
| Léxico (VADER) | Puntuaciones elaboradas manualmente, maneja la negación | `tweet_search.py` |

El siguiente paso — cubierto en `embeddings/` — es usar los vectores de GloVe como **inicialización**
de la capa de embeddings de un clasificador, permitiendo que el modelo los ajuste finamente para la tarea específica.